# Stage 3 — Vision Transformer on CIFAR-10

Implementing the full ViT architecture from *An Image is Worth 16x16 Words*.

Two new components extend the mini Transformer from Stage 2:
- **Patch Embedding**: splits the image into fixed-size patches and projects each to `d_model`
- **CLS token + Positional Encoding**: a learnable classification token prepended to the sequence;
  learnable position vectors added to each patch so the model knows spatial order

The CLS token output (position 0) is used for classification — it aggregates global image
information through attention across all patch positions.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

## 1. Attention + Transformer Block (from Stages 1–2)

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    # [*, N, d_k] @ [*, d_k, N] -> [*, N, N]
    scores  = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    weights = torch.softmax(scores, dim=-1)  # [*, N, N]
    return weights @ V                       # [*, N, d_v]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        B, N, D = X.shape

        Q = self.W_Q(X)  # [B, N, D]
        K = self.W_K(X)  # [B, N, D]
        V = self.W_V(X)  # [B, N, D]

        # [B, N, D] -> [B, num_heads, N, d_k]
        Q = Q.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        K = K.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)

        out = scaled_dot_product_attention(Q, K, V)  # [B, num_heads, N, d_k]

        # [B, num_heads, N, d_k] -> [B, N, D]
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.W_O(out)  # [B, N, D]


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, X):
        X = self.norm1(X + self.attn(X))  # [B, N, d_model]
        X = self.norm2(X + self.ffn(X))   # [B, N, d_model]
        return X

## 2. Patch Embedding

CIFAR-10 images are `[3, 32, 32]`. With `patch_size=4`, the image is divided into
a 8x8 grid of non-overlapping patches — 64 patches total, each covering `4x4x3 = 48` values.

A `Conv2d` with `kernel_size=patch_size, stride=patch_size` performs the split and projection
in one step: each convolution application covers exactly one patch (no overlap),
producing one `d_model`-dimensional vector per patch.

```
[B, 3, 32, 32]  --(Conv2d)--> [B, d_model, 8, 8]  --(flatten+transpose)--> [B, 64, d_model]
```

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # kernel_size == stride -> non-overlapping patches, one output per patch
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, X):
        X = self.proj(X)                                        # [B, d_model, h, w]
        X = X.reshape(X.shape[0], X.shape[1], self.num_patches) # [B, d_model, num_patches]
        X = X.transpose(-2, -1)                                 # [B, num_patches, d_model]
        return X

In [ ]:
B = 2
X = torch.randn(B, 3, 32, 32)

patch_embed = PatchEmbedding(img_size=32, patch_size=4, in_channels=3, d_model=256)
out = patch_embed(X)
print(out.shape)  # [2, 64, 256]

## 3. Vision Transformer

The full ViT forward pass:

1. Embed patches: `[B, 3, H, W]` → `[B, num_patches, d_model]`
2. Prepend CLS token: `[B, num_patches+1, d_model]`
3. Add positional encodings (same shape, added element-wise)
4. Pass through `num_layers` Transformer blocks
5. Extract CLS token output at position 0: `[B, d_model]`
6. Classify: `[B, num_classes]`

Both `cls_token` and `pos_embed` are `nn.Parameter` — learnable tensors updated by the optimizer.

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model, num_heads, num_layers, num_classes):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed   = nn.Parameter(torch.randn(1, num_patches + 1, d_model))  # +1 for CLS

        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm        = nn.LayerNorm(d_model)
        self.classifier  = nn.Linear(d_model, num_classes)

    def forward(self, X):
        B = X.shape[0]

        X   = self.patch_embed(X)                  # [B, num_patches, d_model]
        cls = self.cls_token.expand(B, -1, -1)     # [B, 1, d_model]
        X   = torch.cat([cls, X], dim=1)           # [B, num_patches+1, d_model]
        X   = X + self.pos_embed                   # [B, num_patches+1, d_model]

        for block in self.blocks:
            X = block(X)

        X       = self.norm(X)
        cls_out = X[:, 0]               # [B, d_model] — CLS token at position 0
        return self.classifier(cls_out) # [B, num_classes]

In [ ]:
model_test = ViT(img_size=32, patch_size=4, in_channels=3,
                 d_model=256, num_heads=8, num_layers=6, num_classes=10)
X = torch.randn(2, 3, 32, 32)
print(model_test(X).shape)  # [2, 10]
del model_test

## 4. Training on CIFAR-10

Using FP16 mixed precision (`autocast` + `GradScaler`) to stay within 6 GB VRAM.

Transformers are sensitive to learning rate — `3e-4` works well here.
`1e-3` causes early instability where loss stalls at ~2.2 (near random chance).

In [ ]:
IMG_SIZE    = 32
PATCH_SIZE  = 8
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2  # 64
D_MODEL     = 256
NUM_HEADS   = 8
NUM_LAYERS  = 6
NUM_CLASSES = 10
BATCH_SIZE  = 64
LR          = 3e-4
EPOCHS      = 10

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
train_set    = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_set     = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = ViT(IMG_SIZE, PATCH_SIZE, 3, D_MODEL, NUM_HEADS, NUM_LAYERS, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
scaler    = GradScaler('cuda')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        with autocast('cuda'):
            loss = criterion(model(imgs), labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
save_path = f'models/vit_cifar10_p{PATCH_SIZE}_e{EPOCHS}.pth'
torch.save(model.state_dict(), save_path)
print(f'Saved: {save_path}')

In [ ]:
model.eval()
correct = total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds   = model(imgs).argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

print(f"Test accuracy: {correct / total * 100:.2f}%")